# AIMO 3 - Qwen2.5-Math-72B Submission

**Strategy:** Use the strongest open math model with SC-TIR inference.

- Model: Qwen2.5-Math-72B-Instruct (72 billion parameters)
- Hardware: H100 GPU
- Method: Self-Consistency with Tool-Integrated Reasoning

In [ ]:
# Install vLLM - ignore dependency warnings (they don't affect functionality)
import warnings
warnings.filterwarnings('ignore')

!pip install vllm --quiet 2>&1 | grep -v "dependency\|incompatible\|WARNING\|ERROR" || true
!pip install transformers accelerate --quiet 2>&1 | grep -v "dependency\|incompatible\|WARNING\|ERROR" || true

print("✓ Installation complete - dependency warnings are safe to ignore")

In [ ]:
import os
import re
import signal
import subprocess
import tempfile
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional

import pandas as pd
import torch
from tqdm import tqdm

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
print(f"Submission mode: {IS_SUBMISSION}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# ============= CONFIGURATION =============
# Optimized for H100 + 72B model

@dataclass
class Config:
    # Model - using strongest open math model
    model_id: str = "Qwen/Qwen2.5-Math-72B-Instruct"
    
    # SC-TIR parameters (tuned for 72B)
    num_samples: int = 16        # Width: fewer needed with smarter model
    num_generations: int = 4     # Depth: code execution rounds
    
    # Sampling
    temperature: float = 0.8     # Diversity for self-consistency
    max_new_tokens: int = 2048   # Max generation length
    
    # Code execution
    timeout: int = 10            # Seconds per code block

config = Config()
print(f"Model: {config.model_id}")
print(f"Samples: {config.num_samples}, Generations: {config.num_generations}")

In [ ]:
# ============= PYTHON CODE EXECUTOR =============

class PythonREPL:
    """Execute Python code safely with timeout."""
    
    def __init__(self, timeout: int = 10):
        self.timeout = timeout
        self.imports = """import math
import numpy as np
import sympy as sp
from sympy import *
from fractions import Fraction
from itertools import permutations, combinations, product
from functools import reduce
import sys
sys.set_int_max_str_digits(100000)
"""
    
    @contextmanager
    def time_limit(self, seconds):
        def handler(*_):
            raise TimeoutError(f"Code timed out after {seconds}s")
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
    
    def __call__(self, code: str) -> tuple[bool, str]:
        # Block dangerous operations
        for forbidden in ["subprocess", "os.system", "__import__", "eval(", "exec(", "open("]:
            if forbidden in code:
                return False, f"{forbidden} is not allowed"
        
        full_code = self.imports + code
        
        # Ensure output is printed
        lines = full_code.strip().split("\n")
        last_line = lines[-1].split("#")[0].strip()
        if last_line and "print(" not in last_line and "=" not in last_line:
            lines[-1] = f"print({last_line})"
        full_code = "\n".join(lines)
        
        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "solution.py")
            with open(path, "w") as f:
                f.write(full_code)
            
            try:
                with self.time_limit(self.timeout):
                    result = subprocess.run(
                        ["python3", path],
                        capture_output=True, text=True, timeout=self.timeout
                    )
                    if result.returncode == 0:
                        output = result.stdout.strip()
                        return True, output[:1000] if len(output) > 1000 else output
                    else:
                        error = result.stderr.strip()
                        # Extract just the error message
                        lines = error.split("\n")
                        return False, lines[-1][:500] if lines else "Unknown error"
            except TimeoutError as e:
                return False, str(e)
            except Exception as e:
                return False, str(e)[:200]

executor = PythonREPL(config.timeout)

In [ ]:
# ============= ANSWER EXTRACTION =============

def extract_boxed(text: str) -> Optional[str]:
    """Extract answer from \\boxed{} or \\fbox{}."""
    # Find last \boxed or \fbox
    for marker in ["\\boxed", "\\fbox"]:
        idx = text.rfind(marker)
        if idx >= 0:
            break
    else:
        return None
    
    # Find matching braces
    i, depth = idx, 0
    start = None
    while i < len(text):
        if text[i] == "{":
            if depth == 0:
                start = i + 1
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None


def normalize_answer(text: str) -> str:
    """Clean up answer text."""
    if not text:
        return "-1"
    
    # Remove LaTeX formatting
    text = re.sub(r"\\text\{([^}]*)\}", r"\1", text)
    text = re.sub(r"\\mathrm\{([^}]*)\}", r"\1", text)
    text = re.sub(r"\\mathbf\{([^}]*)\}", r"\1", text)
    
    # Remove common junk
    for r in ["$", "%", ",", " ", "\\,", "\\;", "\\!"]:
        text = text.replace(r, "")
    
    return text.strip()


def to_int(text: str) -> int:
    """Convert to integer in valid range 0-99999."""
    try:
        # Handle fractions
        if "/" in text and text.replace("/", "").replace("-", "").isdigit():
            parts = text.split("/")
            val = int(parts[0]) // int(parts[1])
        else:
            val = round(float(text))
        
        # Clamp to valid range
        if val < 0:
            return -1  # Invalid
        return min(val, 99999)
    except:
        return -1

In [ ]:
# ============= LOAD MODEL =============

from vllm import LLM, SamplingParams

print(f"Loading {config.model_id}...")
print("This may take a few minutes for 72B model.")

llm = LLM(
    model=config.model_id,
    tensor_parallel_size=torch.cuda.device_count(),
    dtype="bfloat16",       # Better precision for H100
    trust_remote_code=True,
    max_model_len=4096,     # Limit context to save memory
    gpu_memory_utilization=0.90,
)

tokenizer = llm.get_tokenizer()
print(f"Model loaded! Using {torch.cuda.device_count()} GPU(s)")

In [ ]:
# ============= PROMPTING =============

SYSTEM_PROMPT = """You are an expert mathematician solving olympiad-level problems.
Think step by step. Use Python code for calculations when needed.
Always put your final numerical answer in \\boxed{}."""

def make_prompt(problem: str) -> str:
    """Create chat-formatted prompt."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Solve this problem:\n\n{problem}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def execute_code_blocks(text: str) -> str:
    """Find and execute Python code blocks."""
    blocks = re.findall(r"```python(.*?)```", text, re.DOTALL)
    if not blocks:
        return ""
    
    # Execute last code block
    success, output = executor(blocks[-1])
    return output

In [ ]:
# ============= SC-TIR SOLVER =============

def solve(problem: str) -> int:
    """Solve using Self-Consistency with Tool-Integrated Reasoning."""
    
    sampling = SamplingParams(
        temperature=config.temperature,
        max_tokens=config.max_new_tokens,
        stop=["```output"],
        include_stop_str_in_output=True,
    )
    
    # Initialize candidates
    prompt = make_prompt(problem)
    candidates = [prompt] * config.num_samples
    
    # SC-TIR loop
    for step in range(config.num_generations):
        # Generate continuations
        outputs = llm.generate(candidates, sampling)
        
        new_candidates = []
        for output in outputs:
            text = output.prompt + output.outputs[0].text
            
            # Execute code if present
            if text.rstrip().endswith("```output"):
                code_result = execute_code_blocks(text)
                text = text.rstrip() + f"\n{code_result}\n```\n\n"
            
            new_candidates.append(text)
        
        candidates = new_candidates
    
    # Extract and vote on answers
    answers = []
    for text in candidates:
        boxed = extract_boxed(text)
        if boxed:
            normalized = normalize_answer(boxed)
            val = to_int(normalized)
            if val >= 0:
                answers.append(val)
    
    # Majority vote
    if not answers:
        print("  WARNING: No valid answers extracted, returning 0")
        return 0
    
    winner = Counter(answers).most_common(1)[0]
    print(f"  Answer distribution: {Counter(answers).most_common(5)}")
    print(f"  Selected: {winner[0]} (votes: {winner[1]}/{len(answers)})")
    
    return winner[0]

In [ ]:
# ============= MAIN LOOP =============

if IS_SUBMISSION:
    # Kaggle submission mode
    import kaggle_evaluation.aimo_3_inference_server
    
    def predict(df):
        problem_id = df["id"].iloc[0]
        problem = df["problem"].iloc[0]
        print(f"\n{'='*60}")
        print(f"Problem {problem_id}")
        print(f"{'='*60}")
        
        answer = solve(problem)
        print(f"Final answer: {answer}")
        
        return pd.DataFrame({"id": df["id"], "answer": [answer]})
    
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()

else:
    # Local testing on reference problems
    print("\nRunning local test on reference problems...\n")
    
    ref_path = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
    test_df = pd.read_csv(ref_path)
    
    results = []
    for idx, row in test_df.iterrows():
        print(f"\n{'='*60}")
        print(f"Problem {idx+1}/{len(test_df)}: {row['id']}")
        print(f"{'='*60}")
        
        pred = solve(row["problem"])
        true = row["answer"]
        correct = (pred == true)
        
        results.append({"id": row["id"], "true": true, "pred": pred, "correct": correct})
        print(f"\nTrue: {true}, Pred: {pred} → {'✓' if correct else '✗'}")
    
    # Summary
    results_df = pd.DataFrame(results)
    accuracy = results_df["correct"].mean()
    
    print(f"\n{'='*60}")
    print(f"RESULTS: {results_df['correct'].sum()}/{len(results_df)} = {accuracy:.1%}")
    print(f"{'='*60}")